In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO
import torch


In [2]:
import psutil
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1e9:.1f} GB")
print(f"Available RAM: {ram.available / 1e9:.1f} GB")

Total RAM: 14.9 GB
Available RAM: 4.0 GB


In [3]:
import os

dataset_path = r'C:\Users\Aditya\BadmintonML\Badminton-Detection-2'

# Check if folder exists at all
print(f"Folder exists: {os.path.exists(dataset_path)}")

# List everything in it
if os.path.exists(dataset_path):
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            print(f"{subindent}{file}")

Folder exists: True
Badminton-Detection-2/
  roboflow.zip


In [4]:
import os

dataset_path = r'C:\Users\Aditya\BadmintonML\Badminton-Detection-2'
zip_path = os.path.join(dataset_path, 'roboflow.zip')

# Check file size
size = os.path.getsize(zip_path)
print(f"File size: {size / 1e6:.1f} MB")

# Read first few bytes to identify format
with open(zip_path, 'rb') as f:
    header = f.read(16)
    print(f"File header (hex): {header.hex()}")
    print(f"File header (raw): {header}")

File size: 1224.7 MB
File header (hex): 504b03041400000800006807b85c1eb8
File header (raw): b'PK\x03\x04\x14\x00\x00\x08\x00\x00h\x07\xb8\\\x1e\xb8'


In [5]:
import tarfile
import os

dataset_path = r'C:\Users\Aditya\BadmintonML\Badminton-Detection-2'
zip_path = os.path.join(dataset_path, 'roboflow.zip')

# Try tar extraction
if tarfile.is_tarfile(zip_path):
    print("It's a tar file — extracting...")
    with tarfile.open(zip_path, 'r:*') as tar:
        tar.extractall(dataset_path)
    print("Extraction complete")
    print(f"Contents: {os.listdir(dataset_path)}")
else:
    print("Not a tar file either")
    print(f"File size: {os.path.getsize(zip_path) / 1e6:.1f} MB")

Not a tar file either
File size: 1224.7 MB


In [3]:
from ultralytics import YOLO
import cv2
import os

# ---- Models ----
shuttle_model = YOLO('Models/last_shuttle.pt')
court_model = YOLO('Models/best_court.pt')
player_model = YOLO('yolo11s.pt')

# ---- Paths ----
INPUT_VIDEO = 'LZJvsVA_Vid.mp4'
video_name = os.path.splitext(INPUT_VIDEO)[0]
OUTPUT_VIDEO = f'runs/detect/{video_name}_analyzed.avi'
os.makedirs('output', exist_ok=True)

# ---- Video Setup ----
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}x{height} @ {fps}fps — {total_frames} frames")

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'XVID'),
    fps,
    (width, height)
)

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---- Player Detection ----
    player_results = player_model.track(
        frame,
        classes=[0],        # person only
        conf=0.5,
        tracker='bytetrack.yaml',
        pesist=True,
        verbose=False,
        device=0
    )

    # ---- Shuttle Detection ----
    shuttle_results = shuttle_model.track(
        frame,
        classes=[0],
        conf=0.2,
        tracker='bytetrack.yaml',
        persist=True,
        verbose=False,
        device=0
    )

    # ---- Court Keypoints ----
    if frame_count % 30 == 0 or cached_court is None:
        cached_court = court_model.predict(
            frame,
            conf=0.25,
            verbose=False,
            device=0
    )

    # ---- Draw Everything ----
    # ---- Draw Everything ----
    # Layer 1 — players
    annotated = player_results[0].plot()

    # Layer 2 — shuttle on top
    annotated = shuttle_results[0].plot(img=annotated)

    # Layer 3 — court keypoints on top
    annotated = cached_court[0].plot(img=annotated)

    # ---- Frame Counter Overlay ----
    cv2.putText(annotated,
               f'Frame: {frame_count}/{total_frames}',
               (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX,
               0.7, (255, 255, 255), 2)

    out.write(annotated)
    frame_count += 1

    if frame_count % 100 == 0:
        print(f"Processed {frame_count}/{total_frames} frames ({frame_count/total_frames*100:.1f}%)")

cap.release()
out.release()
print(f"\nDone — saved to {OUTPUT_VIDEO}")

Video: 0x0 @ 0fps — 0 frames

Done — saved to runs/detect/LZJvsVA_Vid_analyzed.avi
